# Time Series Forecasting with ARIMA Models In Python [Part 1]

Run all cells in order in a fresh Python 3.11 kernel. Complete original datasets are pinned and SHA-256 verified. All original gist examples are preserved verbatim in nonexecuting appendices. Code and matching explanations were corrected with user approval. Stock examples are historical educational exercises, not investment advice. Same-day cross-company regression is distinguished from the past-only lag forecast. No keys or GPU. All writes use an owned temporary workspace.

In [ ]:
import os,sys,subprocess,hashlib,urllib.request,urllib.parse,warnings,random
from pathlib import Path
from tempfile import TemporaryDirectory
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','pandas==3.1.0','numpy==2.3.5','scipy==1.16.3','matplotlib==3.10.8','statsmodels==0.14.6','pmdarima==2.1.1','scikit-learn==1.9.1','joblib==1.6.0','certifi==2026.10.23'],check=True)
import certifi
os.environ['SSL_CERT_FILE']=certifi.where()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
np.random.seed(42);random.seed(42)
pd.set_option('display.max_rows',12);pd.set_option('display.max_columns',12)
plt.rcParams.update({'figure.dpi':80,'figure.figsize':(10,5),'font.size':10})
workspace=TemporaryDirectory(prefix='tdb-time-series-11-');work=Path(workspace.name);previous_cwd=Path.cwd()
source_revision='65d4ae5d3436be3c0a921a3dcbda1b52e3ac8597'
source_files={'Arima Models in Python/candy_production.csv': '6b09ba45d482a23459499b98006d2e4911f31a06ded14ac2a1192dd5279c9056', 'Arima Models in Python/amazon_close.csv': 'f73498d0e1efa5cf9936bd4d2f810259c67ef39b3436f406b194de778fdf40ec', 'Arima Models in Python/co2.csv': 'a777e6a6000ff2dbf9bc12ff652a73a59b3d9c9e459f4330c3fd2574c6768b5c', 'Arima Models in Python/earthquakes.csv': '0d6e92b04f0337d13ea6055785032189763652ee4ec4bffbd80961964d6bbff9', 'Machine Learning For Time Series/prices.csv': 'd6fbdd4831ec8fd5884a690740d38bfc9dff44cfde0decc78ba122e70f58ebf6', 'Machine Learning For Time Series/preprocessed_prices.csv': '23e45fb2fbba7cc93ede87805546df86cba5531e82cc8e3cbdb4d63ddab3b176'}
for relative,digest in source_files.items():
    url='https://raw.githubusercontent.com/youssefHosni/Time-Series-Analysis/'+source_revision+'/'+urllib.parse.quote(relative)
    with urllib.request.urlopen(url,timeout=120)as response:payload=response.read()
    assert hashlib.sha256(payload).hexdigest()==digest,relative
    (work/Path(relative).name).write_bytes(payload)
os.chdir(work)
def checked_fit(model):
    result=model.fit(disp=False,maxiter=1000)
    if not result.mle_retvals.get('converged',False):
        result=model.fit(start_params=result.params,method='powell',disp=False,maxiter=2000)
    assert result.mle_retvals.get('converged',False),'Model did not converge'
    assert np.isfinite(result.params).all() and np.isfinite(result.aic)
    return result
print('Python:',sys.version.split()[0],'| pandas:',pd.__version__,'| NumPy:',np.__version__)
print('Six complete original datasets verified at pinned revision. No API keys or GPU.')

In [ ]:
# Load in the time series
candy = pd.read_csv('candy_production.csv', 
            index_col='date',
            parse_dates=True).sort_index().asfreq('MS')
# change the plot style into fivethirtyeight 
plt.style.use('fivethirtyeight')

# Plot and show the time series on axis ax1
fig, ax1 = plt.subplots()
candy.plot(ax=ax1, figsize=(12,10))
plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 1 completed.')

In [ ]:
# Split the data into a train and test set
candy_train = candy.loc[:'2006']
candy_test = candy.loc['2007':]

# Create an axis
fig, ax = plt.subplots()

# Plot the train and test sets on the axis ax
candy_train.plot(ax=ax, figsize=(12,10))
candy_test.plot(ax=ax)
plt.title('train - test split of the monthly production of candy in US')
plt.xlabel('Date')
plt.ylabel('Production')
plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 2 completed.')

In [ ]:
from statsmodels.tsa.stattools import adfuller
results = adfuller(candy.iloc[:,0])
print(results)
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 3 completed.')

In [ ]:
# Calculate the first difference and drop the nans
candy_diff = candy.diff()
candy_diff = candy_diff.dropna()

# Run test and print
result_diff = adfuller(candy_diff.iloc[:,0])
print(result_diff)

for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 4 completed.')

In [ ]:
from statsmodels.tsa.arima_process import arma_generate_sample
ar_coefs = [1, -0.5] 
ma_coefs = [1, 0.2]
y = arma_generate_sample(ar_coefs, ma_coefs, nsample=100, scale=0.5)
plt.plot(y);plt.title('Simulated ARMA(1,1), 100 observations');plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 5 completed.')

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
# Instantiate model object
model = ARIMA(y, order=(1,0,1))
# Fit model
results = model.fit(method_kwargs={'maxiter':1000})
assert results.mle_retvals['converged']

for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 6 completed.')

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
# Instantiate model object
model = ARIMA(y, order=(1,0,1))
# Fit model
results = model.fit(method_kwargs={'maxiter':1000})
assert results.mle_retvals['converged']
print(results.summary())
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 7 completed.')

In [ ]:
df = pd.DataFrame({'level':np.cumsum(y)})
diff_model = ARIMA(df['level'].diff().dropna(),order=(1,0,1),trend='n').fit(method_kwargs={'maxiter':1000})
assert diff_model.mle_retvals['converged']
diff_forecast = np.asarray(diff_model.forecast(steps=20))
mean_forecast = np.cumsum(diff_forecast) + df.iloc[-1,0]
plt.plot(df.index,df['level'],label='Observed integrated series')
plt.plot(np.arange(len(df),len(df)+20),mean_forecast,label='Integrated forecast')
plt.legend();plt.show()
print('Twenty reconstructed level forecasts:',mean_forecast)
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 8 completed.')

In [ ]:
amazon = pd.read_csv('amazon_close.csv',
                     index_col='date',
                     parse_dates=True).sort_index()
amazon.plot()
plt.title('Amazon stock price change with time')
plt.ylabel('Stock price')
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 9 completed.')

In [ ]:
from statsmodels.tsa.stattools import adfuller

# Run Dicky-Fuller test
result = adfuller(amazon.iloc[:,0])

# Print test statistic
print('The test stastics:', result[0])

# Print p-value
print("The p-value:",result[1])
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 10 completed.')

In [ ]:
# take the first diff
amazon_diff = amazon.diff()
amazon_diff = amazon_diff.dropna()

# Run Dicky-Fuller test
result = adfuller(amazon_diff.iloc[:,0])

# Print test statistic
print('The test stastics:', result[0])

# Print p-value
print("The p-value:",result[1])
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 11 completed.')

In [ ]:
amazon_diff = amazon.diff().dropna()
manual_fit = ARIMA(amazon_diff['close'].to_numpy(),order=(1,0,1),trend='n').fit(method_kwargs={'maxiter':1000})
assert manual_fit.mle_retvals['converged']
manual_steps = np.asarray(manual_fit.forecast(20))
manual_levels = amazon['close'].iloc[-1] + np.cumsum(manual_steps)
print(manual_fit.summary())
print('Twenty forecasts after integrating differences:',manual_levels)
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 12 completed.')

In [ ]:
integrated_fit = ARIMA(amazon['close'].to_numpy(),order=(1,1,1),trend='n').fit(method_kwargs={'maxiter':1000})
assert integrated_fit.mle_retvals['converged']
integrated_prediction = integrated_fit.get_forecast(20)
integrated_levels = np.asarray(integrated_prediction.predicted_mean)
integrated_interval = np.asarray(integrated_prediction.conf_int())
print(integrated_fit.summary())
print('Twenty directly integrated level forecasts:',integrated_levels)
future_steps=np.arange(len(amazon),len(amazon)+20)
plt.plot(np.arange(len(amazon)),amazon['close'],label='Original Amazon closing price')
plt.plot(future_steps,manual_levels,label='Manual integration')
plt.plot(future_steps,integrated_levels,label='ARIMA integration')
plt.fill_between(future_steps,integrated_interval[:,0],integrated_interval[:,1],alpha=.2)
plt.xlabel('Observed trading sessions, followed by forecast steps');plt.ylabel('Stock price');plt.legend();plt.show()
for figure_number in plt.get_fignums():
    display(plt.figure(figure_number))
plt.close('all')
print('Example 13 completed.')

In [ ]:
from numpy.testing import assert_allclose
from scipy.stats import norm
def independent_forecast(result,steps):
    ssm=result.model.ssm
    Z=ssm.design[:,:,0];T=ssm.transition[:,:,0];R=ssm.selection[:,:,0];Q=ssm.state_cov[:,:,0];H=ssm.obs_cov[:,:,0]
    obs=ssm.obs_intercept[:,0];drift=ssm.state_intercept[:,0]
    state=result.filter_results.predicted_state[:,-1].copy()
    covariance=result.filter_results.predicted_state_cov[:,:,-1].copy()
    means=[];variances=[]
    for _ in range(steps):
        means.append(float((Z@state+obs)[0]));variances.append(float((Z@covariance@Z.T+H)[0,0]))
        state=T@state+drift;covariance=T@covariance@T.T+R@Q@R.T
    means=np.asarray(means);variances=np.asarray(variances)
    intervals=np.column_stack([means-norm.ppf(.975)*np.sqrt(variances),means+norm.ppf(.975)*np.sqrt(variances)])
    return means,intervals
from scipy.signal import lfilter
innovations=np.random.RandomState(42).normal(scale=.5,size=100)
assert_allclose(y,lfilter([1,.2],[1,-.5],innovations),atol=1e-12)
assert len(candy)==564 and len(candy_train)==420 and len(candy_test)==144
assert candy_train.index.max()<candy_test.index.min()
assert len(amazon)==1259 and amazon.index.is_monotonic_increasing
assert amazon.index[0].strftime('%Y-%m-%d')=='2014-02-10'
assert_allclose(amazon_diff['close'],np.diff(amazon['close'].to_numpy()),atol=1e-12)
running=float(amazon['close'].iloc[-1]);independent=[]
for step in manual_steps:running+=float(step);independent.append(running)
assert_allclose(manual_levels,independent,atol=1e-10)
means,intervals=independent_forecast(integrated_fit,20)
assert_allclose(integrated_levels,means,atol=1e-9)
assert_allclose(integrated_interval,intervals,atol=1e-9)
assert integrated_fit.mle_retvals['converged'] and manual_fit.mle_retvals['converged']
assert np.isfinite(manual_levels).all() and np.isfinite(integrated_interval).all()
metrics={'candy_rows':len(candy),'candy_train_rows':len(candy_train),'candy_test_rows':len(candy_test),'amazon_rows':len(amazon),'amazon_level_adf_p':float(adfuller(amazon['close'])[1]),'amazon_difference_adf_p':float(adfuller(amazon_diff['close'])[1]),'forecast_steps':20,'manual_last_forecast':float(manual_levels[-1]),'integrated_last_forecast':float(integrated_levels[-1]),'integrated_last_interval':integrated_interval[-1].tolist()}
print('Independent ARMA recurrence, chronological splits, all differences, integrated means and interval checks passed.')
print('METRICS:',metrics)

os.chdir(previous_cwd)
workspace.cleanup()
print('Owned temporary workspace cleaned up.')

## Original source code

### Original block 1

```text
# Load in the time series
candy = pd.read_csv('candy_production.csv', 
            index_col='date',
            parse_dates=True)
# change the plot style into fivethirtyeight 
plt.style.use('fivethirtyeight')

# Plot and show the time series on axis ax1
fig, ax1 = plt.subplots()
candy.plot(ax=ax1, figsize=(12,10))
plt.show()
```

### Original block 2

```text
# Split the data into a train and test set
candy_train = candy.loc[:'2006']
candy_test = candy.loc['2007':]

# Create an axis
fig, ax = plt.subplots()

# Plot the train and test sets on the axis ax
candy_train.plot(ax=ax, figsize=(12,10))
candy_test.plot(ax=ax)
plt.title('train - test split of the monthly production of candy in US')
plt.xlabel('Date')
plt.ylabel('Production')
plt.show()
```

### Original block 3

```text
from statsmodels.tsa.stattools import adfuller
results = adfuller(candy)
print(results)
```

### Original block 4

```text
# Calculate the first difference and drop the nans
candy_diff = candy.diff()
candy_diff = candy_diff.dropna()

# Run test and print
result_diff = adfuller(candy_diff)
print(result_diff)

```

### Original block 5

```text
from statsmodels.tsa.arima_process import arma_generate_sample
ar_coefs = [1, -0.5] 
ma_coefs = [1, 0.2]
y = arma_generate_sample(ar_coefs, ma_coefs, nsample=100, scale=0.5)
```

### Original block 6

```text
from statsmodels.tsa.arima.model import ARIMA
# Instantiate model object
model = ARIMA(y, order=(1,0,1))
# Fit model
results = model.fit()

```

### Original block 7

```text
from statsmodels.tsa.arima.model import ARIMA
# Instantiate model object
model = ARIMA(y, order=(1,0,1))
# Fit model
results = model.fit()
print(results.summary())
```

### Original block 8

```text
from numpy import cumsum
mean_forecast = cumsum(diff_forecast) + df.iloc[-1, 0]
```

### Original block 9

```text
amazon = pd.read_csv('amazon_close.csv',
                     index_col='date',
                     parse_dates=True)
amazon.plot()
plt.title('Amazon stock price change with time')
plt.ylabel('Stock price')
```

### Original block 10

```text
from statsmodels.tsa.stattools import adfuller

# Run Dicky-Fuller test
result = adfuller(amazon)

# Print test statistic
print('The test stastics:', result[0])

# Print p-value
print("The p-value:",result[1])
```

### Original block 11

```text
# take the first diff
amazon_diff = amazon.diff()
amazon_diff.dropna(inplace=True)

# Run Dicky-Fuller test
result = adfuller(amazon_diff)

# Print test statistic
print('The test stastics:', result[0])

# Print p-value
print("The p-value:",result[1])
```

### Original block 12

```text
from statsmodels.tsa.arima.model import ARIMA
# Instantiate model object
model = ARIMA(amazon_diff, order=(1,0,1))
# Fit model
results = model.fit()
print(results.summary())
```

### Original block 13

```text
from statsmodels.tsa.arima.model import ARIMA
# Instantiate model object
model = ARIMA(amazon, order=(1,1,1))
# Fit model
results = model.fit()
print(results.summary())
```